# 02 exploratory data analysis

**Owner:** Daniel. Original notebook cells: 59, 61, 62, 63.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

### EDA Visualizations on Cleaned Features

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load the cleaned dataset
clean_df = pd.read_csv(CLEAN_OUTPUT_PATH)

print(f"Cleaned dataset loaded for EDA. Shape: {clean_df.shape}")

# Plot 1: Lead Time Distribution by Target
plt.figure(figsize=(10, 5))
sns.histplot(data=clean_df, x='LeadTime_Days', hue='No-show', bins=30, kde=True)
plt.title('Distribution of Lead Time by No-Show Status')
plt.xlim(0, 100)
plt.show()

# Plot 2: Feature Correlation Matrix
plt.figure(figsize=(10, 8))
corr_cols = ['Age', 'Scholarship', 'Hipertension', 'Diabetes', 'Alcoholism', 'Handcap', 'SMS_received', 'LeadTime_Days', 'target_no_show']
sns.heatmap(clean_df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:

import pandas as pd
import numpy as np

# Load the original and cleaned datasets
raw_path = "/Volumes/workspace/default/eye_style_team2/raw/KaggleV2-May-2016.csv"
clean_path = "/Volumes/workspace/default/eye_style_team2/clean/cleaned_appointments_features.csv"

raw_df = pd.read_csv(raw_path)
clean_df = pd.read_csv(clean_path)

print("=== DATASET COMPARISON ===")
print(f"Original records: {len(raw_df):,}")
print(f"Cleaned records: {len(clean_df):,}")
print(f"Records removed: {len(raw_df) - len(clean_df):,}")

print("\n=== ORIGINAL DATA QUALITY ===")
print("Missing values:")
print(raw_df.isna().sum())

print("\nExact duplicate rows:", raw_df.duplicated().sum())
print("Duplicate appointment IDs:", raw_df["AppointmentID"].duplicated().sum())
print("Negative ages:", (raw_df["Age"] < 0).sum())

# Parse dates and check for invalid booking lead times
scheduled = pd.to_datetime(raw_df["ScheduledDay"], errors="coerce", utc=True)
appointment = pd.to_datetime(raw_df["AppointmentDay"], errors="coerce", utc=True)

lead_days = (
    appointment.dt.normalize() - scheduled.dt.normalize()
).dt.days

print("\nInvalid scheduling dates:", scheduled.isna().sum())
print("Invalid appointment dates:", appointment.isna().sum())
print("Negative booking lead times:", (lead_days < 0).sum())

print("\n=== REMOVED RECORDS ===")
removed = raw_df.loc[
    ~raw_df["AppointmentID"].isin(clean_df["AppointmentID"])
]

display(removed)

print("\nReasons for removal:")
print("Negative age:", (removed["Age"] < 0).sum())
print(
    "Negative lead time:",
    (lead_days.loc[removed.index] < 0).sum()
)


In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Use Daniel's cleaned dataset
df = clean_df.copy()

# Verify required columns
required = [
    "Age", "LeadTime_Days",
    "SMS_received", "target_no_show"
]
assert all(col in df.columns for col in required)

# 1. No-show rate by booking lead time
lead_bins = [-1, 0, 2, 7, 14, 30, 60, float("inf")]
lead_labels = [
    "Same day", "1-2 days", "3-7 days",
    "8-14 days", "15-30 days",
    "31-60 days", "61+ days"
]

df["LeadTime_Group"] = pd.cut(
    df["LeadTime_Days"],
    bins=lead_bins,
    labels=lead_labels
)

# 2. No-show rate by age group
age_bins = [-1, 17, 29, 44, 59, 74, float("inf")]
age_labels = [
    "0-17", "18-29", "30-44",
    "45-59", "60-74", "75+"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=age_bins,
    labels=age_labels
)

analyses = [
    ("LeadTime_Group", "No-show Rate by Booking Lead Time"),
    ("Age_Group", "No-show Rate by Age Group"),
    ("SMS_received", "No-show Rate by SMS Status")
]

for column, title in analyses:
    results = (
        df.groupby(column, observed=True)["target_no_show"]
        .agg(["mean", "count"])
    )

    results["No_Show_Rate"] = results["mean"] * 100

    print(f"\n=== {title} ===")
    display(results[["count", "No_Show_Rate"]].round(2))

    ax = results["No_Show_Rate"].plot(
        kind="bar",
        figsize=(9, 5),
        rot=25
    )

    ax.set_title(title)
    ax.set_ylabel("No-show rate (%)")
    ax.set_xlabel("")
    ax.set_ylim(0, max(40, results["No_Show_Rate"].max() * 1.15))

    plt.tight_layout()
    plt.show()